# Python: Image to PDf

In [21]:

from PIL import Image
from pathlib import Path


SUPPORTED_FORMATS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".bmp",
    ".gif",
    ".tiff",
    ".tif",
}


def prepare_image(image_path):
    """
    Open an image and convert it to RGB
    for reliable PDF creation.
    """

    image = Image.open(image_path)

    # For animated GIF/WebP, use the first frame
    if getattr(image, "is_animated", False):
        image.seek(0)

    # Convert images with transparency or other modes
    # into RGB with a white background.
    if image.mode != "RGB":

        background = Image.new(
            "RGB",
            image.size,
            "white"
        )

        if "A" in image.mode:
            background.paste(
                image,
                mask=image.getchannel("A")
            )
        else:
            background.paste(image)

        image = background

    else:
        image = image.copy()

    return image


def single_image_to_pdf(image_path):
    """
    Convert one image to PDF.
    The PDF is saved beside the original image.
    """

    image_path = Path(image_path)

    if not image_path.exists():
        print("\nError: Image file does not exist.")
        return

    if not image_path.is_file():
        print("\nError: The provided path is not a file.")
        return

    if image_path.suffix.lower() not in SUPPORTED_FORMATS:
        print("\nError: Unsupported image format.")
        return

    # Same folder + same filename + .pdf
    output_path = image_path.with_suffix(".pdf")

    try:

        image = prepare_image(image_path)

        image.save(
            output_path,
            "PDF",
            resolution=100.0
        )

        print("\nPDF created successfully!")
        print(f"Saved at: {output_path}")

    except Exception as error:

        print(f"\nError: {error}")


def folder_images_to_pdf(folder_path):
    """
    Convert all supported images inside a folder
    into one PDF.

    The PDF is saved inside the same folder.
    """

    folder_path = Path(folder_path)

    if not folder_path.exists():
        print("\nError: Folder does not exist.")
        return

    if not folder_path.is_dir():
        print("\nError: The provided path is not a folder.")
        return

    image_paths = [
        file
        for file in folder_path.iterdir()
        if (
            file.is_file()
            and file.suffix.lower() in SUPPORTED_FORMATS
        )
    ]

    # Sort images alphabetically
    image_paths.sort(
        key=lambda file: file.name.lower()
    )

    if not image_paths:
        print("\nNo supported images found in the folder.")
        return

    print(f"\nFound {len(image_paths)} image(s).")
    print()

    converted_images = []

    for index, image_path in enumerate(
        image_paths,
        start=1
    ):

        try:

            print(
                f"[{index}/{len(image_paths)}] "
                f"Processing: {image_path.name}"
            )

            image = prepare_image(image_path)

            converted_images.append(image)

        except Exception as error:

            print(
                f"Skipped: {image_path.name} "
                f"({error})"
            )

    if not converted_images:
        print("\nNo images could be processed.")
        return

    # PDF will be created inside the input folder
    output_path = folder_path / "merged_images.pdf"

    try:

        first_image = converted_images[0]

        remaining_images = converted_images[1:]

        first_image.save(
            output_path,
            "PDF",
            save_all=True,
            append_images=remaining_images,
            resolution=100.0
        )

        print("\nPDF created successfully!")
        print(f"Saved at: {output_path}")
        print(f"Total pages: {len(converted_images)}")

    except Exception as error:

        print(f"\nError while creating PDF: {error}")


def main():

    print("=" * 55)
    print("          IMAGE TO PDF CONVERTER")
    print("=" * 55)

    print("\nIs it single image to PDF?")
    print("1. Yes")
    print("2. No")

    choice = input("\nEnter your choice (1/2): ").strip()

    # ==========================================
    # SINGLE IMAGE
    # ==========================================

    if choice == "1":

        image_path = input(
            "\nEnter the image path: "
        ).strip().strip('"')

        single_image_to_pdf(image_path)

    # ==========================================
    # MULTIPLE IMAGES
    # ==========================================

    elif choice == "2":

        folder_path = input(
            "\nEnter the folder path containing images: "
        ).strip().strip('"')

        folder_images_to_pdf(folder_path)

    # ==========================================
    # INVALID CHOICE
    # ==========================================

    else:

        print("\nInvalid choice.")
        print("Please run the program again and choose 1 or 2.")


if __name__ == "__main__":
    main()



          IMAGE TO PDF CONVERTER

Is it single image to PDF?
1. Yes
2. No

Found 25 image(s).

[1/25] Processing: 1.jpg
[2/25] Processing: 10.jpg
[3/25] Processing: 11.jpg
[4/25] Processing: 12.jpg
[5/25] Processing: 13.jpg
[6/25] Processing: 14.jpg
[7/25] Processing: 15.jpg
[8/25] Processing: 16.jpg
[9/25] Processing: 17.jpg
[10/25] Processing: 18.jpg
[11/25] Processing: 19.jpg
[12/25] Processing: 2.jpg
[13/25] Processing: 20.jpg
[14/25] Processing: 21.jpg
[15/25] Processing: 22.jpg
[16/25] Processing: 23.jpg
[17/25] Processing: 24.jpg
[18/25] Processing: 25.jpg
[19/25] Processing: 3.jpg
[20/25] Processing: 4.jpg
[21/25] Processing: 5.jpg
[22/25] Processing: 6.jpg
[23/25] Processing: 7.jpg
[24/25] Processing: 8.jpg
[25/25] Processing: 9.jpg

PDF created successfully!
Saved at: T:\IBA books\word smart 1\merged_images.pdf
Total pages: 25
